In [1]:
import sys
sys.path.append('../')

In [2]:
from sqlalchemy import inspect
import pandas as pd
from datetime import datetime
from src.connections import get_postgres_engine, get_snowflake_engine, get_snowflake_engine2

In [3]:

snowflake_engine = get_snowflake_engine()
postgres_engine = get_postgres_engine()

In [4]:
with postgres_engine.connect() as conn:
    print("connected")

connected


In [9]:
#inspecting tables from postgres engine

tables = inspect(postgres_engine).get_table_names()
tables

['categories',
 'products',
 'customers',
 'orders',
 'coupons',
 'order_items',
 'payments',
 'shipping',
 'inventory',
 'reviews']

In [12]:
#creating dictionary for query

def create_query(table_query):
    query = {}
    for t in tables:
        query[t] = (f"select * from {t}")
    return query
    
    

In [13]:
result = create_query(tables)
result

{'categories': 'select * from categories',
 'products': 'select * from products',
 'customers': 'select * from customers',
 'orders': 'select * from orders',
 'coupons': 'select * from coupons',
 'order_items': 'select * from order_items',
 'payments': 'select * from payments',
 'shipping': 'select * from shipping',
 'inventory': 'select * from inventory',
 'reviews': 'select * from reviews'}

In [11]:
#query

In [14]:
#standardizing table name for raw schema (prefix "stg_")

table_name = {}
for key,value in result.items():
    new = f"stg_{key}"
    table_name[new] = value
    


In [16]:
#table_name

In [15]:
#Read data from database 

In [16]:
dataframe = {}
for table, value in table_name.items():
    dataframe[table] = pd.read_sql(value,postgres_engine)

In [17]:
#dataframe['stg_reviews']

In [18]:
#writing to snowflake pd.to_sql(engine, table index =False)

for table, value in dataframe.items():
    value.to_sql(table,snowflake_engine,if_exists = 'replace', index = False)
print(f"loaded{table} sucessfully")

loadedstg_reviews sucessfully


In [19]:
#Inspecting tables from snowflake before transformation

In [20]:
snow_tables = inspect(snowflake_engine).get_table_names()

In [21]:
#snow_tables

In [22]:
#standardizing table names to upper case 

query1 = {}
for t in snow_tables:
    query1[t] = (f"select * from {t.upper()}")
    

In [23]:
#query1

In [24]:
#query1

In [25]:
# extracting data from snowflake raw schema for tranformation

dataframe2 = {}
for table, value in query1.items():
    dataframe2[table] = pd.read_sql(value,snowflake_engine)

In [26]:
 #dataframe2['stg_customers']

In [27]:
#df1

In [28]:
#from ydata_profiling import ProfileReport


In [29]:
customers = dataframe2['stg_customers']

In [30]:
#dataframe2['stg_customers'].info()

In [31]:
customers.shape

(128, 14)

In [32]:
#df1['last_name']

In [33]:
customers['full_name'] =dataframe2['stg_customers']['first_name']+" "+dataframe2['stg_customers']['last_name']

In [34]:
customers.drop(['first_name','last_name'],axis =1 , inplace =True)


In [35]:
customers.columns

Index(['id', 'email', 'phone', 'address', 'city', 'state', 'country',
       'postal_code', 'password', 'is_active', 'created_at', 'updated_at',
       'full_name'],
      dtype='object')

In [36]:
dataframe2['stg_customers'] = dataframe2['stg_customers'][['id','full_name','email','phone','address','city','state','country','postal_code','is_active','created_at','updated_at']]

In [37]:
#df1['city']=df1['city'].fillna('Not Given')

In [38]:
 products = dataframe2['stg_products']


In [39]:
#products

In [40]:
products.columns

Index(['id', 'category_id', 'name', 'slug', 'description', 'price',
       'image_url', 'is_active', 'created_at', 'updated_at'],
      dtype='object')

In [41]:
 products.drop(['slug','image_url'],axis =1,inplace = True)

In [42]:

 #dataframe2['stg_products']

In [43]:
 order_items = dataframe2['stg_order_items']


In [44]:
#order_items

In [45]:
categories = dataframe2['stg_categories']
categories.columns

Index(['id', 'name', 'slug', 'is_active', 'created_at', 'updated_at'], dtype='object')

In [46]:
categories.drop(['slug'],axis = 1, inplace =True)
#dataframe2['stg_categories']

In [47]:
coupons = dataframe2['stg_coupons']


In [48]:
coupons.rename(columns={'discount_pct':'discount_percent'}, inplace = True)

In [49]:
#dataframe2['stg_coupons']

In [50]:
inventory = dataframe2['stg_inventory']


In [51]:
orders = dataframe2['stg_orders']


In [52]:
payment = dataframe2['stg_payments']


In [53]:
reviews = dataframe['stg_reviews']


In [54]:
#reviews.info()

In [55]:
shipping = dataframe2['stg_shipping']


In [56]:
#reviews

In [57]:
#dataframe2

In [58]:
#prefixing silver layer table

transformed = {}
for table, value in dataframe2.items():
    new = f"trnsfrmed_{table.replace('stg_','')}"
    transformed[new] = value

In [59]:
#transformed

In [60]:
#snowflake engine activation
snowflake_eng2 = get_snowflake_engine2()

In [61]:
#loading transformed layer back to snowflake

for table, value in transformed.items():
     value.to_sql(table, snowflake_eng2, index = False, if_exists = 'replace')

In [62]:

Business Questions to Answer

How much revenue are we making?
How many orders do we have?
How many products are on discount
Who are our trusted customers (based on coupon or other business logic)
which part of the world do they reside? 
What’s the average order value?
Which products make good reviews
How do discounts affect revenue?

SyntaxError: invalid character '’' (U+2019) (3329571791.py, line 8)

In [63]:
pd.set_option('display.max.rows', 567)

In [ ]:
 
#transformed['trnsfrmed_orders'][['coupon_id','subtotal','discount_amount','total']]

In [64]:

customer1 = transformed['trnsfrmed_customers']
orders1 = transformed['trnsfrmed_orders']
products = transformed['trnsfrmed_products']
coupons = transformed['trnsfrmed_coupons']
payments = transformed['trnsfrmed_payments']
reviews = transformed['trnsfrmed_reviews']
inventory =transformed['trnsfrmed_inventory']
order_items =transformed['trnsfrmed_order_items']
categories =transformed['trnsfrmed_categories']
shipping =transformed['trnsfrmed_shipping']

In [ ]:
#orders1

In [65]:
#how many orders?

orde = int(orders1['id'].count())
print(f"{orde} No of orders have been placed so far {datetime.now().date()}")

567 No of orders have been placed so far 2026-04-03


In [66]:
#total expected revenue

total_revenue = int(orders1['total'].sum())
total_revenue

217760

In [67]:
#mismatch in sum of discount_amount and total
mask = orders1['total'].astype(int) != (orders1['subtotal'] - orders1['discount_amount']).astype(int)
print(orders1.loc[mask, ['discount_amount', 'subtotal', 'total']])

    discount_amount  subtotal  total
59            145.0    289.99  145.0
61            105.0    209.99  105.0


In [68]:
# how many orders has been shipped so far
shipped_orders = orders1[orders1['status'] == 'shipped']

nos = int(shipped_orders['id'].count())

print(f"total of {nos} orders shipped as of {datetime.now().date()}")

total of 105 orders shipped as of 2026-04-03


In [69]:
#calculating revenue generated based on shipped orders

revenue = int(shipped_orders['subtotal'].sum())-int(shipped_orders['discount_amount'].sum())
print(f"total revenue generated from shipped items = {revenue}")
               


total revenue generated from shipped items = 40336


In [ ]:
#how many products are on discount?



In [ ]:
coupons.head(2)

In [ ]:
#Find products with discount
#products.head(2)

In [71]:
discounted_orders = orders[orders['coupon_id'].notna()]
#discounted_orders

In [72]:
discounted_products = discounted_orders.merge(
    order_items,
    left_on="id",
    right_on="order_id"
)
discounted_products

,id_x,customer_id,coupon_id,status,subtotal,discount_amount,total,notes,created_at_x,updated_at_x,id_y,order_id,product_id,quantity,unit_price,total_price,created_at_y,updated_at_y
0,209,1,3.0,processing,899.97,449.99,449.99,None,2026-03-07 15:38:48.245834+00:00,2026-03-07 15:38:48.245834+00:00,634,209,1,3,299.99,899.97,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
1,211,62,3.0,cancelled,854.96,427.48,427.48,None,2026-02-14 17:35:48.647552+00:00,2026-02-14 17:35:48.647552+00:00,636,211,1,2,299.99,599.98,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
2,211,62,3.0,cancelled,854.96,427.48,427.48,None,2026-02-14 17:35:48.647552+00:00,2026-02-14 17:35:48.647552+00:00,637,211,8,1,42.00,42.00,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
3,211,62,3.0,cancelled,854.96,427.48,427.48,None,2026-02-14 17:35:48.647552+00:00,2026-02-14 17:35:48.647552+00:00,638,211,9,2,55.00,110.00,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
4,211,62,3.0,cancelled,854.96,427.48,427.48,None,2026-02-14 17:35:48.647552+00:00,2026-02-14 17:35:48.647552+00:00,639,211,13,1,22.99,22.99,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
5,211,62,3.0,cancelled,854.96,427.48,427.48,None,2026-02-14 17:35:48.647552+00:00,2026-02-14 17:35:48.647552+00:00,640,211,4,1,79.99,79.99,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
6,212,54,1.0,pending,537.94,53.79,484.15,Test create per pick.,2026-02-12 16:04:48.993197+00:00,2026-02-12 16:04:48.993197+00:00,641,212,13,1,22.99,22.99,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
7,212,54,1.0,pending,537.94,53.79,484.15,Test create per pick.,2026-02-12 16:04:48.993197+00:00,2026-02-12 16:04:48.993197+00:00,642,212,11,3,65.00,195.00,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
8,212,54,1.0,pending,537.94,53.79,484.15,Test create per pick.,2026-02-12 16:04:48.993197+00:00,2026-02-12 16:04:48.993197+00:00,643,212,6,2,39.99,79.98,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00
9,212,54,1.0,pending,537.94,53.79,484.15,Test create per pick.,2026-02-12 16:04:48.993197+00:00,2026-02-12 16:04:48.993197+00:00,644,212,4,3,79.99,239.97,2026-03-11 02:48:42.266872+00:00,2026-03-11 02:48:42.266872+00:00


In [73]:
# to get the number of products on discount
num_discounted_products = discounted_products["product_id"].nunique()
num_discounted_products


15

In [75]:
# to see the products 
discounted_products_list = discounted_products['product_id'].unique()
discounted_products_list

array([ 1,  8,  9, 13,  4, 11,  6,  3,  2,  7, 12,  5, 15, 10, 14])

In [77]:
product_review = reviews.merge(
    products,
    left_on='product_id',
    right_on='id'
)

In [ ]:
#grouping reviews by product_id
product_reviews = product_review.groupby('product_id').agg({
    "RATING": "mean",
    "ID_y": "count"   
}).reset_index()